In [1]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'])
import os, gc, shutil
import pandas as pd
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import polars as pl

In [2]:
CHUNK_SIZE     = 2000000

PROCESSED_DATA_DIR = '/kaggle/input/datasets/b22dckh072/file05'
TRAIN_PATH = os.path.join(PROCESSED_DATA_DIR, 'train_interactions.parquet')
META_PATH  = os.path.join(PROCESSED_DATA_DIR, 'filtered_metadata.parquet')
CAND_PATH  = os.path.join(PROCESSED_DATA_DIR, 'candidates_phase2.parquet')
TEST_PATH = os.path.join(PROCESSED_DATA_DIR, 'test_interactions.parquet')
FEAT_OUT   = os.path.join('/kaggle/working/features.parquet')

In [3]:
print("Bước 1: Nạp bản phác thảo dữ liệu (Lazy Scan)...")
lf_train = pl.scan_parquet(TRAIN_PATH)
lf_meta = pl.scan_parquet(META_PATH)
lf_cands = pl.scan_parquet(CAND_PATH)

print("Bước 2: Tính toán đặc trưng thống kê (Phục hồi biến Popularity)...")
user_stats = lf_train.group_by('mapped_user_id').agg([
    pl.len().cast(pl.Float32).alias('user_total_actions'),
    pl.col('rating').mean().cast(pl.Float32).alias('user_avg_rating_given')
])

item_stats = lf_train.group_by('mapped_item_id').agg([
    pl.len().cast(pl.Float32).alias('item_total_sales'),
    pl.col('rating').mean().cast(pl.Float32).alias('item_actual_avg_rating'),
    pl.col('verified_purchase').cast(pl.Float32).sum().alias('item_verified_sales'),
    pl.col('helpful_vote').cast(pl.Float32).sum().alias('item_raw_helpful_votes')
]).with_columns([
    (pl.col('item_verified_sales') / pl.col('item_total_sales')).fill_null(0.0).alias('item_verified_ratio'),
    (pl.col('item_raw_helpful_votes') + 1.0).log().alias('item_log_helpful_votes')
]).drop(['item_verified_sales', 'item_raw_helpful_votes']) 

print("Bước 3: Tối ưu Meta Data & Làm sạch cột Giá...")
store_counts = lf_meta.group_by('store').agg(pl.len().cast(pl.Float32).alias('store_popularity'))

lf_meta = (
    lf_meta.with_columns([
        # Sửa lỗi chuỗi rỗng khi parse giá tiền
        pl.col('price').cast(pl.Utf8)
          .str.replace_all(r'[^0-9.]', '')
          .replace("", None) 
          .cast(pl.Float32, strict=False)
          .fill_null(0.0), 

        pl.col('categories').cast(pl.Utf8)
          .str.replace_all(r"\[|\]|'|\"", "") 
          .str.split(',')
          .list.get(2) 
          .str.strip_chars()
          .fill_null('Unknown')
          .cast(pl.Categorical)
          .alias('main_category')
    ])
    .join(store_counts, on='store', how='left')
    .with_columns(pl.col('store_popularity').fill_null(0.0))
    .drop(['store']) # Đã giữ lại rating_number để xếp hạng
)

print("Bước 4: Đánh nhãn & CHUẨN BỊ CONTEXT CHO LAMBDARANK...")
lf_test = pl.scan_parquet(TEST_PATH)
lf_val = lf_test.filter((pl.col('mapped_user_id') % 2) == 0)
valid_users = lf_val.select('mapped_user_id').unique()

lf_cands_val = lf_cands.join(valid_users, on='mapped_user_id', how='inner')

lf_labels = (
    lf_cands_val.select(['mapped_user_id', 'mapped_item_id'])
    .join(
        lf_val.select(['mapped_user_id', 'mapped_item_id']).unique().with_columns(pl.lit(1).alias('label').cast(pl.Int8)),
        on=['mapped_user_id', 'mapped_item_id'],
        how='left'
    )
    .with_columns(pl.col('label').fill_null(0).cast(pl.Int8))
)

df_all_labels = lf_labels.collect(engine="streaming")
df_positives = df_all_labels.filter(pl.col('label') == 1)

# Chỉ giữ lại những người dùng CÓ ÍT NHẤT 1 nhãn 1 (Bắt buộc để Gradient không bị 0)
valid_train_users = df_positives.select('mapped_user_id').unique()

df_labels = (
    df_all_labels
    .join(valid_train_users, on='mapped_user_id', how='inner')
    .sort('mapped_user_id')
)

print(f"Tổng số lượng user hợp lệ để huấn luyện: {valid_train_users.height:,}")
print(f"Tổng số mẫu dương tính (Nhãn 1 - Hit): {df_positives.height:,}")
print(f"Tổng số mẫu huấn luyện đã tối ưu (Dòng): {df_labels.height:,}")

print("Bước 5: Nối Đặc trưng và CHUYỂN ĐỔI SANG THỨ HẠNG CỤC BỘ (LOCAL RANK)...")
df_meta_mem = lf_meta.collect()
df_user_stats_mem = user_stats.collect()
df_item_stats_mem = item_stats.collect()

lf_ranks = lf_cands.select(['mapped_user_id', 'mapped_item_id', 'sasrec_rank', 'lightgcn_rank'])

total_rows = df_labels.height
TEMP_DIR = "feat_chunks_temp"
os.makedirs(TEMP_DIR, exist_ok=True)

for start_idx in range(0, total_rows, CHUNK_SIZE):
    end_idx = min(start_idx + CHUNK_SIZE, total_rows)
    print(f"-> Đang xử lý khối {start_idx:,} đến {end_idx:,}...")
    
    chunk_df = df_labels.slice(start_idx, CHUNK_SIZE)
    unique_users = chunk_df['mapped_user_id'].unique().to_list()
    df_ranks_chunk = lf_ranks.filter(pl.col('mapped_user_id').is_in(unique_users)).unique(subset=['mapped_user_id', 'mapped_item_id']).collect()
    
    chunk_processed = (
        chunk_df.lazy()
        .join(df_ranks_chunk.lazy(), on=['mapped_user_id', 'mapped_item_id'], how='left')
        .join(df_meta_mem.lazy(), on='mapped_item_id', how='left')
        .join(df_user_stats_mem.lazy(), on='mapped_user_id', how='left')
        .join(df_item_stats_mem.lazy(), on='mapped_item_id', how='left')
        .with_columns([
            ((201.0 - pl.col('sasrec_rank').cast(pl.Float32)).clip(lower_bound=0.0)).fill_null(0.0).alias('sasrec_score'),
            ((201.0 - pl.col('lightgcn_rank').cast(pl.Float32)).clip(lower_bound=0.0)).fill_null(0.0).alias('lightgcn_score'),
            
            # 1. Trám các giá trị Null trước để tính Rank không bị lỗi
            pl.col('rating_number').fill_null(0.0).cast(pl.Float32),
            pl.col('item_total_sales').fill_null(0.0).cast(pl.Float32),
            pl.col('item_log_helpful_votes').fill_null(0.0).cast(pl.Float32),
            pl.col('store_popularity').fill_null(0.0).cast(pl.Float32),

            pl.col('user_total_actions').fill_null(0.0),
            pl.col('user_avg_rating_given').fill_null(3.0),
            pl.col('item_actual_avg_rating').fill_null(3.0),
            pl.col('item_verified_ratio').fill_null(0.0)
        ])
        # ==============================================================================
        # 2. TUYỆT CHIÊU LOCAL RANK: Đưa mọi con số khổng lồ về thứ hạng (Từ 1 đến 300)
        # ==============================================================================
        .with_columns([
            pl.col('item_total_sales').rank(descending=True).over('mapped_user_id').alias('item_total_sales'),
            pl.col('rating_number').rank(descending=True).over('mapped_user_id').alias('rating_number'),
            pl.col('item_log_helpful_votes').rank(descending=True).over('mapped_user_id').alias('item_log_helpful_votes'),
            pl.col('store_popularity').rank(descending=True).over('mapped_user_id').alias('store_popularity')
        ])
        .drop(['sasrec_rank', 'lightgcn_rank'])
        .collect()
    )
    
    chunk_processed.write_parquet(f"{TEMP_DIR}/chunk_{start_idx}.parquet")
    
    del chunk_df, df_ranks_chunk, chunk_processed, unique_users
    gc.collect()

print("-> Đang hợp nhất các khối thành file Features hoàn chỉnh...")
pl.scan_parquet(f"{TEMP_DIR}/*.parquet").with_columns([
    pl.col('main_category').cast(pl.Utf8)
]).sink_parquet(FEAT_OUT)

shutil.rmtree(TEMP_DIR)
del df_labels, lf_train, lf_meta, lf_cands, user_stats, item_stats
del df_meta_mem, df_user_stats_mem, df_item_stats_mem
gc.collect()

print(f"Hoàn tất! File dữ liệu huấn luyện an toàn được lưu tại: {FEAT_OUT}")

Bước 1: Nạp bản phác thảo dữ liệu (Lazy Scan)...
Bước 2: Tính toán đặc trưng thống kê (Phục hồi biến Popularity)...
Bước 3: Tối ưu Meta Data & Làm sạch cột Giá...
Bước 4: Đánh nhãn & CHUẨN BỊ CONTEXT CHO LAMBDARANK...
Tổng số lượng user hợp lệ để huấn luyện: 80,591
Tổng số mẫu dương tính (Nhãn 1 - Hit): 92,682
Tổng số mẫu huấn luyện đã tối ưu (Dòng): 24,177,004
Bước 5: Nối Đặc trưng và CHUYỂN ĐỔI SANG THỨ HẠNG CỤC BỘ (LOCAL RANK)...
-> Đang xử lý khối 0 đến 2,000,000...
-> Đang xử lý khối 2,000,000 đến 4,000,000...
-> Đang xử lý khối 4,000,000 đến 6,000,000...
-> Đang xử lý khối 6,000,000 đến 8,000,000...
-> Đang xử lý khối 8,000,000 đến 10,000,000...
-> Đang xử lý khối 10,000,000 đến 12,000,000...
-> Đang xử lý khối 12,000,000 đến 14,000,000...
-> Đang xử lý khối 14,000,000 đến 16,000,000...
-> Đang xử lý khối 16,000,000 đến 18,000,000...
-> Đang xử lý khối 18,000,000 đến 20,000,000...
-> Đang xử lý khối 20,000,000 đến 22,000,000...
-> Đang xử lý khối 22,000,000 đến 24,000,000...
-> Đ

In [4]:
# print("-> Đang hợp nhất các khối thành file Features hoàn chỉnh...")
# pl.scan_parquet(f"{TEMP_DIR}/*.parquet").with_columns([
#     pl.col('main_category').cast(pl.Utf8)
# ]).sink_parquet(FEAT_OUT)

# # 1. Xuất Từ điển User
# USER_FEAT_OUT = '/kaggle/working/user_features.parquet'
# df_user_stats_mem.write_parquet(USER_FEAT_OUT)
# print(f"-> Đã lưu User Features: {df_user_stats_mem.height:,} người dùng.")

# # 2. Xuất Từ điển Item 
# ITEM_FEAT_OUT = '/kaggle/working/item_features.parquet'
# df_item_features = (
#     df_meta_mem.lazy()
#     .join(df_item_stats_mem.lazy(), on='mapped_item_id', how='left')
#     .with_columns([
#         pl.col('item_total_sales').fill_null(0),
#         pl.col('price').fill_null(0.0), 
#         pl.col('main_category').cast(pl.Utf8).fill_null('Unknown'),
#         pl.col('item_actual_avg_rating').fill_null(3.0),
#         pl.col('item_verified_ratio').fill_null(0.0),
#         pl.col('item_log_helpful_votes').fill_null(0.0)
#     ])
#     .collect()
# )
# df_item_features.write_parquet(ITEM_FEAT_OUT)
# print(f"-> Đã lưu Item Features: {df_item_features.height:,} sản phẩm.")

# shutil.rmtree(TEMP_DIR)
# del df_labels, lf_train, lf_meta, lf_cands, user_stats, item_stats
# del df_meta_mem, df_user_stats_mem, df_item_stats_mem
# gc.collect()

# print(f"\nHoàn tất! File dữ liệu huấn luyện an toàn được lưu tại: {FEAT_OUT}")

In [5]:
import os
import polars as pl

print("=======================================")
print("ĐÁNH GIÁ ĐỘ PHỦ TỐI ĐA (HIT RATE @ 300) CỦA GIAI ĐOẠN CANDIDATE")
print("=======================================")

# Đảm bảo đường dẫn trỏ đúng vào tập Test và file Candidates 300 món
TEST_PATH = '/kaggle/input/datasets/b22dckh072/file05/test_interactions.parquet'
CAND_PATH = '/kaggle/input/datasets/b22dckh072/file05/candidates_phase2.parquet'

print("1. Đang nạp tập Test và lọc riêng tập Test Chính...")
# CHỈ LẤY USER CÓ ID LẺ (% 2 != 0) ĐỂ ĐỐI CHIẾU
df_test = pl.read_parquet(TEST_PATH).filter((pl.col('mapped_user_id') % 2) != 0)
truth_df = df_test.group_by('mapped_user_id').agg(pl.col('mapped_item_id').alias('true_items'))

print("2. Đang nạp lưới 300 ứng cử viên và lọc tương ứng...")
df_cands = pl.read_parquet(CAND_PATH).filter((pl.col('mapped_user_id') % 2) != 0)
preds_df = df_cands.group_by('mapped_user_id').agg(pl.col('mapped_item_id').alias('pred_items'))

print("3. Đang tiến hành đối chiếu...")
eval_df = truth_df.join(preds_df, on='mapped_user_id', how='inner')

hits = eval_df.with_columns(
    pl.col('true_items').list.set_intersection(pl.col('pred_items')).list.len().alias('hit_count')
).filter(pl.col('hit_count') > 0)

hr_300 = hits.height / eval_df.height

print(f"Tổng số User đối chiếu trong tập Test CHÍNH: {eval_df.height:,}")
print(f"Số User được vớt trúng (ít nhất 1 item) trong lưới 300 món: {hits.height:,}")
print(f"-> HIT RATE @ 300 (Giới hạn trần tối đa tập Test): {hr_300:.4f} ({(hr_300*100):.2f}%)")
print("=======================================")

ĐÁNH GIÁ ĐỘ PHỦ TỐI ĐA (HIT RATE @ 300) CỦA GIAI ĐOẠN CANDIDATE
1. Đang nạp tập Test và lọc riêng tập Test Chính...
2. Đang nạp lưới 300 ứng cử viên và lọc tương ứng...
3. Đang tiến hành đối chiếu...
Tổng số User đối chiếu trong tập Test CHÍNH: 516,025
Số User được vớt trúng (ít nhất 1 item) trong lưới 300 món: 81,468
-> HIT RATE @ 300 (Giới hạn trần tối đa tập Test): 0.1579 (15.79%)
